In [ ]:
import pandas as pd
import re
import os
DB_OUTPUT = f"dataset\database_v3_new"
os.makedirs(DB_OUTPUT, exist_ok=True)

In [ ]:
df = pd.read_csv(r"dataset\URA_merged_full_v3_new_removed_outliers.csv")
print(df['Project Name'].nunique())
print(len(df.columns.tolist()))
for col in df.columns:
    print(col)

171
project_idx
month_idx
date
is_post_cutoff
rent_psf_obs
rent_psf_imp
was_observed
condo_name
size_tier
size_label
district
segment
area_sqft_med
avg_psf_hist
n_transactions
Condo_Age_2026
tenure_remaining_years
tenure_freehold_like
tenure_medium_lease_60_80
tenure_more_than_80
tenure_short_lease_lt60
tenure_unknown
Large_Dev_200plus
project_size_small
project_size_medium
project_size_large
Planning Area
Neighbourhood
nearest_mrt_dist
nearest_mrt_name
nearest_bus_stops_dist
n_bus_stops_top3
nearest_supermarkets_dist
n_supermarkets_top3
nearest_parks_dist
n_parks_top3
nearest_clinics_dist
n_clinics_top3
nearest_bank_dist
n_bank_top3
nearest_atms_dist
n_atms_top3
nearest_school_dist
nearest_school_name
Nanyang Primary School
Rosyth School
Henry Park Primary School
Tao Nan School
Raffles Girls' Primary School
St. Hilda's Primary School
Pei Hwa Presbyterian Primary School
Methodist Girls' School (Primary)
Nan Hua Primary School
Chij St. Nicholas Girls' School
Anglo-Chinese School (Primar

In [7]:
df.head()

,project_idx,month_idx,date,is_post_cutoff,rent_psf_obs,rent_psf_imp,was_observed,condo_name,size_tier,size_label,...,nonlanded_private_units_in_the_pipeline_under_construction_share_lag1q,log1p_non_landed_units_launched_in_private_res_projects_with_prereq_for_sale_lag1q,non_landed_units_in_private_res_projects_with_prereq_not_launched_share_lag1q,log1p_unsold_completed_nonlanded_private_units_lag1q,log1p_unsold_non_landed_units_in_launched_private_res_projects_lag1q,assessed_non_landed_private_residential_owned_by_companies_share_lag1q,assessed_non_landed_private_residential_owned_by_pr_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_companies_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_spr_share_lag1q
0,0,0,1999-11,False,NaN,3.769841,False,1 MOULMEIN RISE,SZ3,medium,...,0.459061,9.601368,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250
1,0,1,1999-12,False,NaN,3.771444,False,1 MOULMEIN RISE,SZ3,medium,...,0.459061,9.601368,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250
2,0,2,2000-01,False,NaN,3.832254,False,1 MOULMEIN RISE,SZ3,medium,...,0.466325,9.481664,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812
3,0,3,2000-02,False,NaN,3.941147,False,1 MOULMEIN RISE,SZ3,medium,...,0.466325,9.481664,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812
4,0,4,2000-03,False,NaN,4.054842,False,1 MOULMEIN RISE,SZ3,medium,...,0.466325,9.481664,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812


In [5]:
df['Lease Commencement Date'] = pd.to_datetime(df['Lease Commencement Date'], errors='coerce')
df = df.dropna(subset=['Lease Commencement Date'])
df = df.sort_values('Lease Commencement Date')
df['timestep'] = df['Lease Commencement Date'].rank(method='dense').astype(int) - 1
df['transaction_id'] = [i for i in range(1, len(df) + 1)]
df['project_id'] = df['Project Name'].astype('category').cat.codes

C:\Users\cecel\AppData\Local\Temp\ipykernel_36592\3658964363.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['timestep'] = df['Lease Commencement Date'].rank(method='dense').astype(int) - 1
C:\Users\cecel\AppData\Local\Temp\ipykernel_36592\3658964363.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['transaction_id'] = [i for i in range(1, len(df) + 1)]
C:\Users\cecel\AppData\Local\Temp\ipykernel_36592\3658964363.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

In [6]:
project_df = (
    df[['project_id', 'Project Name']]
    .drop_duplicates()
    .sort_values('project_id')
)

project_df.to_csv(os.path.join(DB_OUTPUT, "project_id.csv"), index=False)

In [7]:
timestep_df = (
    df[['timestep', 'Lease Commencement Date']]
    .drop_duplicates()
    .sort_values('timestep')
)

timestep_df.to_csv(os.path.join(DB_OUTPUT, "timestep.csv"), index=False)

In [8]:
# df['sqft'] = (
#     df['Floor Area (SQFT)']
#     .astype(str)
#     .str.replace(',', '', regex=False)          # remove commas
#     .str.extract(r'(\d+)\s*to\s*(\d+)')         # extract numbers
#     .astype(float)
#     .mean(axis=1)
# )
# df['rent_per_sqft'] = df['Monthly Rent ($)'] / df['sqft']

## V3
df = df.rename(columns={
    "Floor Area (SQFT) mid": "sqft",
    "Rent per SQFT": "rent_per_sqft"
})

In [9]:
transaction_df = (
    df[['transaction_id', 'timestep', 'Lease Commencement Date', 'project_id', 'Project Name', 'No of Bedroom', 'Floor Area (SQFT)', 'sqft', 'Monthly Rent ($)', 'rent_per_sqft']]
    .drop_duplicates()
    .sort_values('transaction_id')
)

transaction_df.to_csv(os.path.join(DB_OUTPUT, "transaction.csv"), index=False)

In [11]:
macro_cols = [
    'timestep',
    'Lease Commencement Date',
    'cpi_all_items_infl_yoy_pct',
    'unemployment_rate_sa_pct',
    'sora_overnight_pct',
    'sora_compounded_3m_pct',
    'sg_govt_bond_yield_10y_pct',
    'sgd_per_usd_logret_12m_pct',
    'ura_private_rental_index_yoy_pct',
    'ura_private_price_index_avg_3seg_yoy_pct',
    'hdb_resale_price_index_yoy_pct',
    'gva_yoy_growth_pct',
    'cpi_all_items_index',
    'unemployment_rate_sa_pct_monthly',
    'sora_overnight_pct_monthly',
    'sora_compounded_3m_pct_monthly',
    'sg_govt_bond_yield_10y_pct_monthly',
    'sgd_per_usd',
    'ura_private_rental_index',
    'ura_private_price_index_ccr',
    'ura_private_price_index_rcr',
    'ura_private_price_index_ocr',
    'hdb_resale_price_index',
    'gva_yoy_growth_pct_monthly',
    'cpi_all_items_infl_mom_pct',
    'cpi_all_items_infl_yoy_pct_monthly',
    'ura_private_rental_index_qoq_pct',
    'ura_private_rental_index_yoy_pct_monthly',
    'ura_private_price_index_ccr_qoq_pct',
    'ura_private_price_index_ccr_yoy_pct',
    'ura_private_price_index_rcr_qoq_pct',
    'ura_private_price_index_rcr_yoy_pct',
    'ura_private_price_index_ocr_qoq_pct',
    'ura_private_price_index_ocr_yoy_pct',
    'hdb_resale_price_index_qoq_pct',
    'hdb_resale_price_index_yoy_pct_monthly',
    'ura_private_price_index_avg_3seg',
    'ura_private_price_index_avg_3seg_qoq_pct',
    'ura_private_price_index_avg_3seg_yoy_pct_monthly',
    'sgd_per_usd_logret_1m_pct',
    'sgd_per_usd_logret_12m_pct_monthly',
    'sora_overnight_pct_chg_bp',
    'sora_compounded_3m_pct_chg_bp',
    'sg_govt_bond_yield_10y_pct_chg_bp',
    'unemployment_rate_sa_chg_pp'
]

macro_feature_cols = macro_cols[2:]
existing_cols = [col for col in macro_cols if col in df.columns]
missing_cols = [col for col in macro_cols if col not in df.columns]

if missing_cols:
    print(f"Warning: Missing columns -> {missing_cols}")

macro_df = (
    df[existing_cols]
    .drop_duplicates()
    .sort_values('timestep')
    .reset_index(drop=True)
)

macro_df.to_csv(os.path.join(DB_OUTPUT, "macro_data.csv"), index=False)

In [10]:
project_cols = [
    'project_id', 'Project Name',
    'onemap_address',
    'latitude',
    'longitude',
    'Street Name',
    'Postal District',
    'Property Type',
    # 'Completion Date', 
    'Planning Region',
    'Planning Area',
    # 'Tenure Length',
    # 'Tenure Start Date',
    'Large_Dev_200plus',
    'Condo_Age_2026',
    'tenure_remaining_years',
    'tenure_freehold_like',
    'tenure_medium_lease_60_80',
    'tenure_more_than_80',
    'tenure_short_lease_lt60',
    'tenure_unknown',
    # 'project_size_small',
    # 'project_size_medium',
    # 'project_size_large',
]

project_df = (
    df[project_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)

project_df.to_csv(os.path.join(DB_OUTPUT, "project.csv"), index=False)

In [12]:
pre_cols = ['project_id', 'Project Name']
school_cols = [
    col for col in df.columns
    if "School" in col or "Institution" in col
]
pre_school_cols = pre_cols + school_cols
school_df = (
    df[pre_school_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)

school_df.to_csv(os.path.join(DB_OUTPUT, "school.csv"), index=False)

In [13]:
pre_cols = ['project_id', 'Project Name']
mrt_cols = [col for col in df.columns if "_MRT" in col]
pre_mrt_cols = pre_cols + mrt_cols
mrt_df = (
    df[pre_mrt_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)

# convert meters → kilometers
mrt_df[mrt_cols] = mrt_df[mrt_cols] / 1000
mrt_df.to_csv(os.path.join(DB_OUTPUT, "mrt.csv"), index=False)

In [14]:
pre_cols = ['project_id', 'Project Name']
dist_cols = [col for col in df.columns if " dist" in col]
pre_dist_cols = pre_cols + dist_cols
dist_df = (
    df[pre_dist_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)
def convert_to_km(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip().lower()
    
    # extract numeric value
    match = re.search(r"[\d\.]+", x)
    if not match:
        return None
    
    value = float(match.group())
    
    # convert based on unit
    if "km" in x:
        return value
    elif "m" in x:
        return value / 1000
    else:
        if value/1000 < 0.01:
            return value
        else:
            return value  # unknown unit
        
for col in dist_cols:
    dist_df[col] = dist_df[col].apply(convert_to_km)
dist_df.to_csv(os.path.join(DB_OUTPUT, "proximity.csv"), index=False)

In [15]:
pre_cols = ['project_id', 'Project Name']
ame_cols = [col for col in df.columns if "Has_" in col]
pre_ame_cols = pre_cols + ame_cols
ame_df = (
    df[pre_ame_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)
# df2 = pd.read_csv(r"dataset\database\amenities.csv")
# ame_df = pd.merge(ame_df, df2, on="Project Name", how="left")
ame_df.to_csv(os.path.join(DB_OUTPUT, "amenities.csv"), index=False)

In [20]:
df_existing = pd.read_csv(r"C:\Users\cecel\OneDrive - National University of Singapore\Master Course Work\Dissertation\Code\My_Code\PGIM-Graph\dataset\database_v3\macro_data.csv")
df_original = pd.read_excel(r"C:\Users\cecel\Downloads\sg_macro_monthly.xlsx")
cols_existing = set(df_existing.columns)
cols_original = set(df_original.columns)

# In original but missing in existing
missing_in_existing = cols_original - cols_existing

# In existing but not in original
extra_in_existing = cols_existing - cols_original

# Common columns
common_cols = cols_existing & cols_original

print("Missing in df_existing:", missing_in_existing)
print("Extra in df_existing:", extra_in_existing)
print("Common columns:", common_cols)

Missing in df_existing: {'month'}
Extra in df_existing: {'sora_overnight_pct_monthly', 'ura_private_price_index_avg_3seg_yoy_pct_monthly', 'sora_compounded_3m_pct_monthly', 'hdb_resale_price_index_yoy_pct_monthly', 'sg_govt_bond_yield_10y_pct_monthly', 'gva_yoy_growth_pct_monthly', 'unemployment_rate_sa_pct_monthly', 'sgd_per_usd_logret_12m_pct_monthly', 'timestep', 'cpi_all_items_infl_yoy_pct_monthly', 'Lease Commencement Date', 'ura_private_rental_index_yoy_pct_monthly'}
Common columns: {'unemployment_rate_sa_pct', 'ura_private_price_index_ocr_qoq_pct', 'sg_govt_bond_yield_10y_pct_chg_bp', 'unemployment_rate_sa_chg_pp', 'ura_private_price_index_rcr_yoy_pct', 'sg_govt_bond_yield_10y_pct', 'sora_overnight_pct', 'hdb_resale_price_index_yoy_pct', 'sora_compounded_3m_pct', 'gva_yoy_growth_pct', 'ura_private_price_index_avg_3seg_yoy_pct', 'ura_private_price_index_ccr_yoy_pct', 'ura_private_price_index_avg_3seg', 'sgd_per_usd_logret_12m_pct', 'hdb_resale_price_index_qoq_pct', 'sgd_per_usd_l

In [24]:
import pandas as pd

df_existing = df_existing.copy()
df_original = df_original.copy()

# -------------------------------
# 1. Standardize dates
# -------------------------------
df_existing["Lease Commencement Date"] = pd.to_datetime(df_existing["Lease Commencement Date"])
df_original["month"] = pd.to_datetime(df_original["month"])

df_existing["Lease Commencement Date"] = (
    df_existing["Lease Commencement Date"].dt.to_period("M").dt.to_timestamp()
)
df_original["month"] = (
    df_original["month"].dt.to_period("M").dt.to_timestamp()
)

# -------------------------------
# 2. Build mapping: date -> timestep
# -------------------------------
timestep_map = df_existing[[
    "Lease Commencement Date", "timestep"
]].drop_duplicates()

# -------------------------------
# 3. Filter df_original by existing date range
# -------------------------------
min_date = timestep_map["Lease Commencement Date"].min()
max_date = timestep_map["Lease Commencement Date"].max()

df_original_filtered = df_original[
    (df_original["month"] >= min_date) &
    (df_original["month"] <= max_date)
].copy()

# -------------------------------
# 4. Merge to bring timestep
# -------------------------------
df_new = df_original_filtered.merge(
    timestep_map,
    left_on="month",
    right_on="Lease Commencement Date",
    how="inner"   # ensures only aligned months remain
)

# -------------------------------
# 5. Clean columns
# -------------------------------
df_new = df_new.drop(columns=["Lease Commencement Date"])  # keep only one date column
df_new = df_new.rename(columns={"month": "Lease Commencement Date"})

# -------------------------------
# 6. Reorder columns: timestep first
# -------------------------------
cols = ["timestep"] + [c for c in df_new.columns if c != "timestep"]
df_new = df_new[cols]

# -------------------------------
# 7. Sort
# -------------------------------
df_new = df_new.sort_values("timestep").reset_index(drop=True)

In [25]:
df_new.to_csv(r"C:\Users\cecel\OneDrive - National University of Singapore\Master Course Work\Dissertation\Code\My_Code\PGIM-Graph\dataset\database_v3\macro_data_v1.csv", index=False)